# Sentinel-2

Use the launch button at the top of this page to open the full notebook in Google Colab. This workflow uses Sentinel-2 imagery in Earth Engine to map fire effects with NDVI, NBR, and delta NBR.


In [ ]:
# %pip install earthengine-api
# %pip install geemap

In [ ]:
import ee
import geemap

In [ ]:
ee.Authenticate()              # Step 1: authenticate your Earth Engine account


In [ ]:
ee.Initialize(project='your-cloud-project')    # Step 2: initialize Earth Engine with your Cloud project ID


https://developers.google.com/earth-engine/datasets/catalog/sentinel-2

![Sentinel notebook figure 1](../images/gee/sentinel-01.png)

![Sentinel notebook figure 2](../images/gee/sentinel-02.png)

## Hands-on workflow


### Task 1: Draw the California wildfire area on the map and choose the pre-/post-fire dates


In [ ]:
Map = geemap.Map(center=[39.8, -121.4], zoom=9)

In [ ]:
Map

In [ ]:
fire_area = Map.user_roi

In [ ]:
# fire_area = ee.Geometry.Rectangle([-118.847309, 33.99575, -118.462022, 34.177725])

In [ ]:
# Define pre- and post-fire image date windows.
before_start, before_end = '2024-12-01', '2025-01-05'
after_start, after_end = '2025-01-15', '2025-02-15'


In [ ]:
tiles_before_fire = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(fire_area).filterDate(before_start, before_end)

In [ ]:
tiles_after_fire = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(fire_area).filterDate(after_start, after_end)

In [ ]:
tiles_before_fire

In [ ]:
tiles_before_fire = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(fire_area).filterDate(before_start, before_end).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

In [ ]:
tiles_before_fire

In [ ]:
tiles_after_fire = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(fire_area).filterDate(after_start, after_end).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

In [ ]:
tiles_after_fire

In [ ]:
tile_before_fire = tiles_before_fire.median()

In [ ]:
tile_before_fire

In [ ]:
Map.addLayer(tile_before_fire, {'bands': ['B4', 'B3', 'B2']}, 'before fire RGB')

In [ ]:
Map

In [ ]:
tile_after_fire = tiles_after_fire.median()

In [ ]:
image_before_fire = tile_before_fire.clip(fire_area)
image_after_fire = tile_after_fire.clip(fire_area)

### Task 3: Compute NDVI and NBR


### NDVI: Normalized Difference Vegetation Index



NDVI = (B8 - B4) / (B8 + B4)

- `B8`: near-infrared (NIR), where vegetation reflects strongly


- `B4`: red, where vegetation absorbs strongly


| NDVI value | Typical surface type |
| ---------- | -------------------- |
| < 0 | Water, snow, cloud, etc. |
| 0-0.2 | Bare soil, desert, built-up surfaces |
| 0.2-0.5 | Grassland, cropland, shrubland |
| 0.5-0.8 | Healthy forest or dense vegetation |


### NBR: Normalized Burn Ratio


NBR  = (B8 - B12) / (B8 + B12)

- `B12`: shortwave infrared 2 (SWIR2), which responds strongly over burned areas


- Healthy vegetation: high NIR and low SWIR, so NBR is high.


- Burned areas: NIR decreases and SWIR increases, so NBR drops sharply.


ΔNBR = NBR_before - NBR_after

| Delta NBR range | Burn severity |
| --------------- | ------------- |
| < 0.1 | Unburned / negligible change |
| 0.1-0.27 | Low severity |
| 0.27-0.44 | Moderate severity |
| > 0.44 | High severity |


In [ ]:
NDVI_before = image_before_fire.normalizedDifference(['B8', 'B4']).rename('NDVI')
NDVI_after = image_after_fire.normalizedDifference(['B8', 'B4']).rename('NDVI')

In [ ]:
NBR_before = image_before_fire.normalizedDifference(['B8', 'B12']).rename('NBR')
NBR_after = image_after_fire.normalizedDifference(['B8', 'B12']).rename('NBR')

In [ ]:
delta_NBR = NBR_before.subtract(NBR_after).rename('deltaNBR')

### Task 4: Display the burned area using delta NBR


In [ ]:
# Visualization parameters
ndvi_vis = {'min': 0, 'max': 0.8, 'palette': ['brown', 'yellow', 'green']}
nbr_vis = {'min': 0, 'max': 1, 'palette': ['white', 'blue', 'black']}
delta_vis = {'min': 0, 'max': 1, 'palette': ['white', 'orange', 'red']}


In [ ]:
# Add layers to the map.
Map.addLayer(NDVI_before, ndvi_vis, 'NDVI Before')
Map.addLayer(NDVI_after, ndvi_vis, 'NDVI After')
Map.addLayer(NBR_before, nbr_vis, 'NBR Before')
Map.addLayer(NBR_after, nbr_vis, 'NBR After')
Map.addLayer(delta_NBR, delta_vis, 'Delta NBR Burned Area')


In [ ]:
Map

### Task 5: Export the analysis result


In [ ]:
geemap.ee_export_image(
    delta_NBR,
    filename='deltaNBR.tif',
    region=fire_area,
    scale=10,  # Sentinel-2 main visible/NIR bands are 10 m resolution.
    file_per_band=False
)


## References

- Google Earth Engine Data Catalog. (n.d.). [Harmonized Sentinel-2 MSI: MultiSpectral Instrument, Level-2A (SR)](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED).
- European Space Agency. (n.d.). [Sentinel-2 mission](https://www.esa.int/Applications/Observing_the_Earth/Copernicus/Sentinel-2).
